# HalfHex — Qwen3-1.7B ONNX Export for QNN Conversion

**One-click notebook.** Run All cells. Downloads the exported ONNX model as a zip at the end.

- **Input:** Qwen/Qwen3-1.7B from HuggingFace
- **Output:** `qwen3-1.7b-onnx.zip` downloaded to your local machine
- **RAM needed:** ~12 GB (Colab free tier is sufficient)
- **Time:** ~5-10 minutes total

In [ ]:
# Cell 1: Install dependencies
!pip install -q torch --index-url https://download.pytorch.org/whl/cpu
!pip install -q transformers optimum[onnxruntime] onnx onnxsim sentencepiece huggingface_hub
# Fix numpy if needed
!pip install -q "numpy<2"
print("✓ Dependencies installed")

In [ ]:
# Cell 2: Verify versions
import torch, transformers, optimum, onnx
print(f"torch:        {torch.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"onnx:         {onnx.__version__}")

from optimum.exporters.onnx import main_export
print("✓ optimum ONNX exporter available")

import psutil
ram_gb = psutil.virtual_memory().total / (1024**3)
print(f"RAM: {ram_gb:.1f} GB")
assert ram_gb >= 10, f"Need >=10 GB RAM, have {ram_gb:.1f} GB. Use Colab Pro or High-RAM runtime."

In [ ]:
# Cell 3: Download Qwen3-1.7B weights
from huggingface_hub import snapshot_download
import time

MODEL_ID = "Qwen/Qwen3-1.7B"
MODEL_DIR = "/content/qwen3-1.7b-hf"

print(f"[{time.strftime('%H:%M:%S')}] Downloading {MODEL_ID}...")
path = snapshot_download(
    repo_id=MODEL_ID,
    local_dir=MODEL_DIR,
    ignore_patterns=["*.gguf", "*.bin", "flax_model*", "tf_model*"],
)
print(f"[{time.strftime('%H:%M:%S')}] ✓ Downloaded to {path}")

import os
total_mb = sum(os.path.getsize(os.path.join(path, f)) for f in os.listdir(path) if os.path.isfile(os.path.join(path, f))) / (1024*1024)
print(f"Total: {total_mb:.0f} MB")

In [ ]:
# Cell 4: Inspect model architecture
from transformers import AutoConfig

config = AutoConfig.from_pretrained(MODEL_DIR)
print("=" * 50)
print("QWEN3-1.7B ARCHITECTURE")
print("=" * 50)
print(f"Layers:      {config.num_hidden_layers}")
print(f"Hidden dim:  {config.hidden_size}")
print(f"Q heads:     {config.num_attention_heads}")
print(f"KV heads:    {config.num_key_value_heads}")
print(f"FFN dim:     {config.intermediate_size}")
print(f"Vocab size:  {config.vocab_size}")
print(f"Head dim:    {config.hidden_size // config.num_attention_heads}")

# KV cache size at 512 context
kv_bytes = config.num_hidden_layers * 2 * config.num_key_value_heads * (config.hidden_size // config.num_attention_heads) * 512 * 2
print(f"\nKV cache @512 (fp16): {kv_bytes/(1024*1024):.1f} MB")

In [ ]:
# Cell 5: Export to ONNX (the main step — takes 3-8 minutes)
import time, gc
from optimum.exporters.onnx import main_export

ONNX_DIR = "/content/qwen3-1.7b-onnx"

print(f"[{time.strftime('%H:%M:%S')}] Starting ONNX export...")
print("This takes 3-8 minutes. Do not interrupt.")
t0 = time.perf_counter()

main_export(
    model_name_or_path=MODEL_DIR,
    output=ONNX_DIR,
    task="text-generation-with-past",
    opset=17,
    device="cpu",
    fp16=False,
    no_post_process=False,
)

elapsed = time.perf_counter() - t0
print(f"\n[{time.strftime('%H:%M:%S')}] ✓ Export complete in {elapsed:.1f}s")
gc.collect()

# List output files
import os
for f in sorted(os.listdir(ONNX_DIR)):
    fpath = os.path.join(ONNX_DIR, f)
    if os.path.isfile(fpath):
        print(f"  {f}: {os.path.getsize(fpath)/(1024*1024):.1f} MB")

In [ ]:
# Cell 6: Simplify ONNX model (removes redundant nodes, const-folds)
import subprocess, os

onnx_files = [f for f in os.listdir(ONNX_DIR) if f.endswith('.onnx')]
print(f"Found ONNX files: {onnx_files}")

for fname in onnx_files:
    src = os.path.join(ONNX_DIR, fname)
    dst = os.path.join(ONNX_DIR, fname.replace('.onnx', '_sim.onnx'))
    print(f"\nSimplifying {fname}...")
    result = subprocess.run(
        ['python', '-m', 'onnxsim', src, dst],
        capture_output=True, text=True, timeout=600
    )
    if result.returncode == 0:
        orig_mb = os.path.getsize(src) / (1024*1024)
        simp_mb = os.path.getsize(dst) / (1024*1024)
        print(f"  ✓ {fname}: {orig_mb:.1f} MB → {simp_mb:.1f} MB ({100*simp_mb/orig_mb:.0f}%)")
    else:
        print(f"  ⚠ onnxsim failed for {fname}: {result.stderr[:200]}")
        print(f"  Using unsimplified model instead.")

In [ ]:
# Cell 7: Profile the ONNX graph
import onnx
from collections import Counter
import os, json

onnx_files = [f for f in os.listdir(ONNX_DIR) if f.endswith('.onnx')]
for fname in onnx_files:
    fpath = os.path.join(ONNX_DIR, fname)
    fsize = os.path.getsize(fpath) / (1024*1024*1024)
    model = onnx.load(fpath)
    op_counts = Counter(n.op_type for n in model.graph.node)
    
    print(f"\n{'='*50}")
    print(f"{fname} ({fsize:.2f} GB)")
    print(f"Total nodes: {len(model.graph.node)}")
    print(f"Inputs:  {[i.name for i in model.graph.input][:5]}...")
    print(f"Outputs: {[o.name for o in model.graph.output][:3]}...")
    print(f"\nOp distribution (top 10):")
    for op, count in op_counts.most_common(10):
        print(f"  {op:30s}: {count}")
    del model

In [ ]:
# Cell 8: Also copy the tokenizer files (needed for on-device inference)
import shutil, os

TOKENIZER_DIR = os.path.join(ONNX_DIR, "tokenizer")
os.makedirs(TOKENIZER_DIR, exist_ok=True)

tokenizer_files = ["tokenizer.json", "tokenizer_config.json", "merges.txt", "vocab.json",
                   "special_tokens_map.json", "tokenizer.model"]
for f in tokenizer_files:
    src = os.path.join(MODEL_DIR, f)
    if os.path.exists(src):
        shutil.copy2(src, TOKENIZER_DIR)
        print(f"  ✓ {f}")
    else:
        print(f"  - {f} (not found, skipping)")

# Also copy config.json
shutil.copy2(os.path.join(MODEL_DIR, "config.json"), ONNX_DIR)
print("  ✓ config.json")

In [ ]:
# Cell 9: Zip everything and trigger download
import shutil, os

ZIP_NAME = "/content/qwen3-1.7b-onnx"
print("Creating zip archive...")
shutil.make_archive(ZIP_NAME, 'zip', '/content', 'qwen3-1.7b-onnx')

zip_path = ZIP_NAME + ".zip"
zip_mb = os.path.getsize(zip_path) / (1024*1024)
print(f"\n✓ Archive: {zip_path} ({zip_mb:.0f} MB)")
print(f"\nDownloading to your machine...")

# Auto-download in Colab
try:
    from google.colab import files
    files.download(zip_path)
    print("✓ Download started! Save to: F:\\Seshu\\Workspace\\MadrasAI Company\\HQQNN-HTP\\models\\")
except ImportError:
    print(f"Not running in Colab. Manually download: {zip_path}")
    print(f"Then extract to: F:\\Seshu\\models\\qwen3-1.7b-onnx\\")

## After Download

1. Save `qwen3-1.7b-onnx.zip` to your machine
2. Extract to `F:\Seshu\models\qwen3-1.7b-onnx\`
3. Tell Claude: **"ONNX model ready"**

Claude will then run:
- `qnn-onnx-converter` (ONNX → QNN INT8)
- `qnn-model-lib-generator` (compile to .so)
- Push to device and benchmark vs llama.cpp 9.05 tok/s